# Data exploration

Query the data already loaded in DSQL. Connection uses the same `data.load.get_engine()` as the loader: IAM token refreshed at engine creation.

In [1]:
import sys
from pathlib import Path

here = Path.cwd()
project_root = here if (here / "pyproject.toml").exists() else here.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
from IPython.display import Markdown

from data.generate_erd import generate_erd

generate_erd(format="png", out_path="docs/erd.png")
Markdown("![ERD](../docs/erd.png)")

In [ ]:
import pandas as pd
from data.load_to_dsql import get_engine

engine = get_engine()
print(engine.url.render_as_string(hide_password=True))

## Row counts

In [ ]:
tables = [
    "supplier", "raw_material",
    "purchase_order", "purchase_order_line",
    "goods_receipt", "goods_receipt_line",
    "document", "supplier_invoice", "supplier_invoice_line",
]
counts = {t: pd.read_sql(f"SELECT COUNT(*) AS n FROM {t}", engine).iloc[0, 0] for t in tables}
pd.Series(counts, name="rows")

## Suppliers and raw materials

In [ ]:
pd.read_sql(
    """
    SELECT city, COUNT(*) AS n
    FROM supplier
    GROUP BY city
    ORDER BY n DESC
    LIMIT 10
    """,
    engine,
)

In [ ]:
pd.read_sql(
    """
    SELECT category, COUNT(*) AS n
    FROM raw_material
    GROUP BY category
    ORDER BY n DESC
    """,
    engine,
)

## Purchase orders

In [ ]:
pd.read_sql(
    """
    SELECT status, COUNT(*) AS n
    FROM purchase_order
    GROUP BY status
    ORDER BY n DESC
    """,
    engine,
)

In [ ]:
monthly = pd.read_sql(
    """
    SELECT DATE_TRUNC('month', order_date) AS month, COUNT(*) AS n
    FROM purchase_order
    GROUP BY month
    ORDER BY month
    """,
    engine,
)
monthly.set_index("month")["n"].plot(
    title="Purchase orders per month: seasonality + YoY growth",
    figsize=(10, 3),
)

In [ ]:
pd.read_sql(
    """
    SELECT s.name, COUNT(*) AS po_count, SUM(po.total_gross_eur) AS total_gross_eur
    FROM purchase_order po
    JOIN supplier s ON s.id = po.supplier_id
    GROUP BY s.name
    ORDER BY total_gross_eur DESC
    LIMIT 10
    """,
    engine,
)

## Supplier invoices: discrepancies for the validation agent

In [ ]:
pd.read_sql(
    """
    SELECT
        si.supplier_invoice_number,
        s.name AS supplier,
        s.iban AS master_iban,
        si.payment_iban
    FROM supplier_invoice si
    JOIN supplier s ON s.id = si.supplier_id
    WHERE si.payment_iban != s.iban
    ORDER BY si.invoice_date DESC
    LIMIT 15
    """,
    engine,
)

In [ ]:
pd.read_sql(
    """
    SELECT
        si.supplier_invoice_number,
        sil.description,
        pol.unit_price_net_eur AS po_price,
        sil.unit_price_net_eur AS invoice_price,
        ROUND(
            (sil.unit_price_net_eur - pol.unit_price_net_eur)
            / pol.unit_price_net_eur * 100, 2
        ) AS drift_pct
    FROM supplier_invoice_line sil
    JOIN purchase_order_line pol ON pol.id = sil.purchase_order_line_id
    JOIN supplier_invoice si ON si.id = sil.supplier_invoice_id
    WHERE sil.unit_price_net_eur > pol.unit_price_net_eur
    ORDER BY drift_pct DESC
    LIMIT 15
    """,
    engine,
)

## One full invoice with its lines (for the agent to extract)

In [ ]:
pd.read_sql(
    """
    SELECT
        si.supplier_invoice_number,
        s.name AS supplier,
        si.invoice_date,
        si.due_date,
        sil.line_number,
        sil.description,
        sil.quantity,
        sil.unit_price_net_eur,
        sil.line_net_eur
    FROM supplier_invoice si
    JOIN supplier s ON s.id = si.supplier_id
    JOIN supplier_invoice_line sil ON sil.supplier_invoice_id = si.id
    WHERE si.supplier_invoice_number = (
        SELECT supplier_invoice_number FROM supplier_invoice LIMIT 1
    )
    ORDER BY sil.line_number
    """,
    engine,
)